In [4]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde
from matplotlib.colors import LinearSegmentedColormap

# Ensure Matplotlib exports text as actual fonts/paths editable in Illustrator
plt.rcParams['svg.fonttype'] = 'none'
plt.rcParams['pdf.fonttype'] = 42

# --- CONFIGURATION ---
file_paths = {
    "SOK1": "SOK1_2021-05-06.csv",
    "SOK1_2": "SOK1_2021-05-03.csv",
    "SOK1_3": "SOK1_2024-01-17.csv",
    "SOK1_manual": "SOK1_manual_measurements.csv",
    "SOK2": "SOK2_2021-07-27.csv",
    "SOK2_2": "SOK2_2021-07-28.csv",
    "SOK2_manual": "SOK2_manual_measurements.csv",
    "SOK5": "SOK5_2021-05-19.csv",
    "SOK5_2": "SOK5_2021-05-26.csv",
    "SOK5_3pt1": "SOK5_2023-02-24_pt1.csv",
    "SOK5_3pt2": "SOK5_2023-02-24_pt2.csv",
    "SOK5_manual": "SOK5_manual_measurements.csv",
    "SOK5_abl": "SOK5abl_2023-11-05_stack0.csv",
    "SOK5_abl_1": "SOK5abl_2023-11-05_stack1.csv"
}

# Define your developmental stages (ordered sequentially)
stages_config = [
    {"name": "stage_II", "Xmin": 15, "Xmax": 23,  "width_px": 200},
    {"name": "stage_III", "Xmin": 23, "Xmax": 30,  "width_px": 300},
    {"name": "stage_IV-VII", "Xmin": 30, "Xmax": 60,  "width_px": 600},
    {"name": "stage_eLR", "Xmin": 60, "Xmax": 130, "width_px": 1250}
    # "All" omitted here since we want the sequential timeline view
]

group_colormaps = {
    "SOK1": LinearSegmentedColormap.from_list("custom_red", [(1,1,1,0), "red"], N=3),
    "SOK2": LinearSegmentedColormap.from_list("custom_orange", [(1,1,1,0), "orange"], N=3),
    "SOK5": LinearSegmentedColormap.from_list("custom_blue", [(1,1,1,0), "blue"], N=3),
    "SOK5abl": LinearSegmentedColormap.from_list("custom_purple", [(1,1,1,0), "purple"], N=3)
}

# Load datasets once into memory
dfs = {}
for key, path in file_paths.items():
    try:
        dfs[key] = pd.read_csv(path, sep=',', decimal='.')
    except FileNotFoundError:
        print(f"Warning: {path} not found. Skipping.")

# --- DYNAMIC FIGURE SIZING ---
height_px = 600
dpi = 100
export_format = "svg"  # Change to "pdf" if preferred

# Sum all widths to find total figure width, adding a bit of padding room between subplots
total_width_px = sum(stage["width_px"] for stage in stages_config)
fig_width = total_width_px / dpi
fig_height = height_px / dpi

# Extract width ratios for GridSpec
width_ratios = [stage["width_px"] for stage in stages_config]

# Create a 1-row grid with dynamically sized column widths
fig, axes = plt.subplots(
    nrows=1, 
    ncols=len(stages_config), 
    figsize=(fig_width, fig_height),
    gridspec_kw={'width_ratios': width_ratios},
    sharey=True # Keeps y-axis scaling identical across all plots
)

# If only 1 stage configuration was passed by accident, wrap it in a list
if len(stages_config) == 1:
    axes = [axes]

fig.patch.set_facecolor("white")
xgrid, ygrid = np.meshgrid(np.linspace(0, 1, 100), np.linspace(0, 0.5, 200))

# --- PROCESSING LOOP ---
for col_idx, stage in enumerate(stages_config):
    ax = axes[col_idx]
    ax.set_facecolor("white")
    
    Xmin = stage["Xmin"]
    Xmax = stage["Xmax"]
    stage_name = stage["name"]
    
    print(f"Plotting Stage: {stage_name}")
    
    z_order = 1
    
    for sok_group, cmap in group_colormaps.items():
        all_x, all_y = [], []
        
        for key, df in dfs.items():
            if key.startswith(sok_group):
                required_columns = [
                    "Width (µm)",
                    "Max 1 X / Columns with Max > 0", "Deviation from 0.5 (Max 1)",
                    "Max 2 X / Columns with Max > 0", "Deviation from 0.5 (Max 2)",
                    "Max 3 X / Columns with Max > 0", "Deviation from 0.5 (Max 3)"
                ]
                if not all(col in df.columns for col in required_columns):
                    continue
                
                X = df["Width (µm)"] / 10
                df_filtered = df[(X >= Xmin) & (X <= Xmax)].copy()
                if df_filtered.empty:
                    continue

                for i in ["1", "2", "3"]:
                    all_x.extend(df_filtered[f"Max {i} X / Columns with Max > 0"])
                    all_y.extend(df_filtered[f"Deviation from 0.5 (Max {i})"])
        
        if len(all_x) > 0 and len(all_y) > 0:
            all_x, all_y = np.array(all_x), np.array(all_y)
            valid_mask = (~np.isnan(all_x)) & (~np.isnan(all_y)) & (~np.isinf(all_x)) & (~np.isinf(all_y))
            all_x, all_y = all_x[valid_mask], all_y[valid_mask]

            if len(all_x) > 1:
                xy = np.vstack([all_x, all_y])
                try:
                    kde = gaussian_kde(xy)
                    grid_positions = np.vstack([xgrid.ravel(), ygrid.ravel()])
                    density_values = kde(grid_positions).reshape(xgrid.shape)

                    # Plot Clouds
                    ax.contourf(xgrid, ygrid, density_values, levels=2, cmap=cmap, alpha=0.33, zorder=z_order)
                    # Plot Outlines
                    ax.contour(xgrid, ygrid, density_values, levels=2, colors="black", linewidths=0.3, zorder=z_order + 15)
                    z_order += 2
                except np.linalg.LinAlgError:
                    pass

    # Clean axes formatting for Illustrator
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 0.5)
    ax.set_xticks([])
    ax.set_yticks([])
    
    # Optional: Adds text titles for stages. 
    # Because of our font path rules, this text remains editable type blocks in Illustrator.
    ax.set_title(stage_name, fontsize=8)

# Remove spacing gaps between subplots entirely so they align perfectly back-to-back
plt.subplots_adjust(wspace=0.05, hspace=0)

# Export the master timeline
output_filename = f"cloudplots_combined_timeline2.{export_format}"
plt.savefig(output_filename, format=export_format, transparent=True, dpi=dpi, bbox_inches='tight', pad_inches=0.1)

print(f"\nSuccess! Saved complete vector canvas to: {output_filename}")
plt.close()

Plotting Stage: stage_II
Plotting Stage: stage_III
Plotting Stage: stage_IV-VII
Plotting Stage: stage_eLR

Success! Saved complete vector canvas to: cloudplots_combined_timeline2.svg
